# Algoritmos de Seleção de Variáveis e Validação Cruzada 

Em um projeto de ciência de dados, há inúmeras variáveis envolvidas: 
- EDA
- Tamanho da base de dados
- Selecionar o modelo
    - supervisionado / não supervisionado
    - classificação / predição
- Para quem você vai reportar
- Uso do conhecimento de domínio (e intuição)
- Conselhos do especialista do domínio
- Tuning de parâmetros
    - elbow function

## E quando já fizemos tudo isso, construímos o modelo, mas a perfomance ainda não está boa?

Uma opção é trabalhar com **Feature Enginnering**, nesse caso, falaremos sobre **seleção de variáveis**

### Feature evaluation
Testes estatísticos podem ser usados para selecionar as features que possuem realacionamento mais forte com a variável que estamos buscando preditar. 

Uma das possibilidades disponíveis no scikit-learn é o **SelectKBest**, que pode ser usada com diferentes testes estatísticos para selecionar um número específico de variáveis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
import warnings
warnings.filterwarnings('ignore')

In [ ]:
file_path = "datasets/diabetes.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
dataframe = pd.read_csv(file_path, names=names)
dataframe.head()

In [ ]:
array = dataframe.values
array

In [ ]:
X = array[:, 0:8]
Y = array[:,8]
print(X)
print(Y)

In [ ]:
k_best_features = SelectKBest(score_func=chi2, k=4)
fit = k_best_features.fit(X, Y)
fit

In [ ]:
np.set_printoptions(precision=2)
# print(fit.scores_)
# print(names)
feature_weights = dict(zip(names, fit.scores_))
feature_weights

In [ ]:
selected_features = fit.transform(X)
print(selected_features[0:5,:])

**Baseline**: usando todas as variáveis

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    Y,
                                                    test_size=0.30,
                                                    random_state=100)

In [ ]:
from sklearn.linear_model import LogisticRegression
logmodel = LogisticRegression()
logmodel.fit(X_train, y_train)
predicitions = logmodel.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predicitions))

In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, predicitions))

Novo modelo, agora com as **variáveis selecionadas**:

In [ ]:
X_train_ft_sel, X_test_ft_sel, y_train_ft_sel, y_test_ft_sel = train_test_split(selected_features,
                                                                                Y, test_size=0.30,
                                                                                random_state=100)

In [ ]:
logmodel_ft_sel = LogisticRegression()
logmodel_ft_sel.fit(X_train_ft_sel, y_train_ft_sel)
predicitions_ft_sel = logmodel_ft_sel.predict(X_test_ft_sel)

In [ ]:
print(classification_report(y_test_ft_sel, predicitions_ft_sel))

In [ ]:
print(confusion_matrix(y_test_ft_sel, predicitions_ft_sel))

Dropamos metade das features do nosso dataset e, ainda assim, houve melhora na performance do modelo

### Recursive Feature Elimination